# Lista 3 - KNN e Árvores de Decisão

**Aluno:** Diego Melo do Nascimento  
**Matrícula:** 603127  
**Disciplina:** Aprendizagem de Máquina (2026.1)  
**Professor:** César Lincoln Cavalcante Mattos

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.tree import DecisionTreeClassifier
from warnings import filterwarnings

filterwarnings("ignore")
np.random.seed(42)

ModuleNotFoundError: No module named 'sklearn'

## Funções Auxiliares

Pré-processamento (normalização Z-score), K-Fold e métricas de classificação binária implementadas do zero.

In [ ]:
def compute_mean(X):
    return np.sum(X, axis=0) / X.shape[0]

def compute_std(X):
    mu = compute_mean(X)
    return np.sqrt(np.sum((X - mu) ** 2, axis=0) / X.shape[0])

def fit_standardize(X):
    mu = compute_mean(X)
    sigma = compute_std(X)
    sigma[sigma == 0] = 1
    return mu, sigma

def apply_standardize(X, mu, sigma):
    return (X - mu) / sigma

def kfold_split(n_samples, k=10, seed=42):
    rng = np.random.default_rng(seed)
    indices = rng.permutation(n_samples)
    folds = np.array_split(indices, k)
    splits = []
    for i in range(k):
        test_idx = folds[i]
        train_idx = np.concatenate([folds[j] for j in range(k) if j != i])
        splits.append((train_idx, test_idx))
    return splits

# --- Métricas binárias implementadas do zero ---

def accuracy(y_true, y_pred):
    return np.mean(y_true == y_pred)

def binary_metrics(y_true, y_pred, pos_class=1):
    tp = np.sum((y_pred == pos_class) & (y_true == pos_class))
    fp = np.sum((y_pred == pos_class) & (y_true != pos_class))
    fn = np.sum((y_pred != pos_class) & (y_true == pos_class))
    tn = np.sum((y_pred != pos_class) & (y_true != pos_class))

    acc = (tp + tn) / (tp + tn + fp + fn)
    recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    f1 = (2 * precision * recall) / (precision + recall) if (precision + recall) > 0 else 0.0

    return acc, recall, precision, f1

def confusion_matrix(y_true, y_pred, classes):
    n = len(classes)
    idx = {c: i for i, c in enumerate(sorted(classes))}
    cm = np.zeros((n, n), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[idx[t], idx[p]] += 1
    return cm

def plot_cm(cm, classes, title, ax):
    cm_pct = cm.astype(float) / cm.sum(axis=1, keepdims=True) * 100
    im = ax.imshow(cm_pct, cmap="Blues", vmin=0, vmax=100)
    ax.set_title(title, fontsize=10)
    ticks = list(sorted(classes))
    ax.set_xticks(range(len(ticks)))
    ax.set_yticks(range(len(ticks)))
    ax.set_xticklabels(ticks)
    ax.set_yticklabels(ticks)
    ax.set_xlabel("Predito")
    ax.set_ylabel("Verdadeiro")
    for i in range(len(ticks)):
        for j in range(len(ticks)):
            color = "white" if cm_pct[i, j] > 50 else "black"
            ax.text(j, i, f"{cm_pct[i,j]:.1f}%\n({cm[i,j]})",
                    ha="center", va="center", color=color, fontsize=8)
    plt.colorbar(im, ax=ax, fraction=0.046)

## Implementação do KNN

KNN implementado do zero com suporte a distância **Euclidiana** e **Mahalanobis**.

- **Euclidiana:** $d(x, x_i) = \|x - x_i\|_2$
- **Mahalanobis:** $d(x, x_i) = \sqrt{(x - x_i)^\top \Sigma^{-1} (x - x_i)}$, onde $\Sigma$ é estimada no conjunto de treino.

In [ ]:
class KNN:
    """K-Nearest Neighbors com distâncias Euclidiana e Mahalanobis."""

    def __init__(self, k=1, distance="euclidean"):
        assert distance in ("euclidean", "mahalanobis")
        self.k = k
        self.distance = distance
        self.X_train = None
        self.y_train = None
        self.VI = None  # inversa da covariância (Mahalanobis)

    def fit(self, X, y):
        self.X_train = X.copy()
        self.y_train = y.copy()
        if self.distance == "mahalanobis":
            cov = np.cov(X.T)
            # regularização para garantir invertibilidade
            cov += np.eye(cov.shape[0]) * 1e-6
            self.VI = np.linalg.inv(cov)
        return self

    def _dist_euclidean(self, x):
        diff = self.X_train - x
        return np.sqrt(np.sum(diff ** 2, axis=1))

    def _dist_mahalanobis(self, x):
        diff = self.X_train - x  # (N, d)
        # d_i = sqrt( diff_i @ VI @ diff_i )
        left = diff @ self.VI          # (N, d)
        dists = np.sqrt(np.sum(left * diff, axis=1))
        return dists

    def _predict_one(self, x):
        if self.distance == "euclidean":
            dists = self._dist_euclidean(x)
        else:
            dists = self._dist_mahalanobis(x)
        nn_idx = np.argsort(dists)[:self.k]
        nn_labels = self.y_train[nn_idx]
        # voto majoritário
        values, counts = np.unique(nn_labels, return_counts=True)
        return values[np.argmax(counts)]

    def predict(self, X):
        return np.array([self._predict_one(x) for x in X])

## Carregamento dos Dados

In [ ]:
data = np.genfromtxt("data/kc2.csv", delimiter=",", skip_header=1)
X = data[:, :-1]
y = data[:, -1].astype(int)

print(f"Amostras: {X.shape[0]} | Features: {X.shape[1]}")
values, counts = np.unique(y, return_counts=True)
for v, c in zip(values, counts):
    print(f"Classe {int(v)}: {int(c)} ({c / y.shape[0] * 100:.1f}%)")

## Questão 1a — Validação Cruzada (10 Folds)

Para cada fold:
1. Normalização Z-score ajustada **apenas** no treino, aplicada ao teste (sem data leakage).
2. KNN treinado e avaliado nas 4 combinações.
3. DecisionTree (scikit-learn) com critérios `gini` e `entropy`.

In [ ]:
def evaluate_kfold(X, y, model_factory, k=10, normalize=True):
    splits = kfold_split(X.shape[0], k=k)
    metrics = {"acc": [], "recall": [], "precision": [], "f1": []}
    all_yt, all_yp = [], []

    for train_idx, test_idx in splits:
        X_tr, X_te = X[train_idx], X[test_idx]
        y_tr, y_te = y[train_idx], y[test_idx]

        if normalize:
            mu, sigma = fit_standardize(X_tr)
            X_tr = apply_standardize(X_tr, mu, sigma)
            X_te = apply_standardize(X_te, mu, sigma)

        model = model_factory()
        model.fit(X_tr, y_tr)
        y_pred = model.predict(X_te)

        acc, rec, prec, f1 = binary_metrics(y_te, y_pred, pos_class=1)
        metrics["acc"].append(acc)
        metrics["recall"].append(rec)
        metrics["precision"].append(prec)
        metrics["f1"].append(f1)
        all_yt.extend(y_te)
        all_yp.extend(y_pred)

    return {mk: np.array(v) for mk, v in metrics.items()}, np.array(all_yt), np.array(all_yp)


models_config = {
    "KNN k=1 Euclidiana":    lambda: KNN(k=1, distance="euclidean"),
    "KNN k=5 Euclidiana":    lambda: KNN(k=5, distance="euclidean"),
    "KNN k=1 Mahalanobis":   lambda: KNN(k=1, distance="mahalanobis"),
    "KNN k=5 Mahalanobis":   lambda: KNN(k=5, distance="mahalanobis"),
    "DecisionTree Gini":     lambda: DecisionTreeClassifier(criterion="gini",    random_state=42),
    "DecisionTree Entropia": lambda: DecisionTreeClassifier(criterion="entropy", random_state=42),
}

print("Executando validacao cruzada (10 folds)...")
results = {}
for name, factory in models_config.items():
    m, yt, yp = evaluate_kfold(X, y, factory)
    results[name] = (m, yt, yp)
    print(f"  ok: {name}")
print("\nConcluido!")

## Questão 1b — Métricas: Média ± Desvio Padrão

In [ ]:
header = f"{'Modelo':<25} {'Acuracia':^20} {'Revocacao':^20} {'Precisao':^20} {'F1-score':^20}"
sep = "-" * len(header)
print("QUESTAO 1 - KNN e ARVORES DE DECISAO")
print(sep)
print(header)
print(sep)
for name, (m, _, _) in results.items():
    acc_s    = f"{np.mean(m['acc']):.4f} +/- {np.std(m['acc']):.4f}"
    rec_s    = f"{np.mean(m['recall']):.4f} +/- {np.std(m['recall']):.4f}"
    prec_s   = f"{np.mean(m['precision']):.4f} +/- {np.std(m['precision']):.4f}"
    f1_s     = f"{np.mean(m['f1']):.4f} +/- {np.std(m['f1']):.4f}"
    print(f"{name:<25} {acc_s:^20} {rec_s:^20} {prec_s:^20} {f1_s:^20}")

## Comparação Gráfica dos Modelos

In [ ]:
model_names = list(results.keys())
short_names = [
    "KNN k=1\nEuclid.", "KNN k=5\nEuclid.",
    "KNN k=1\nMahal.", "KNN k=5\nMahal.",
    "DT\nGini", "DT\nEntropia"
]
metric_keys = ["acc", "recall", "precision", "f1"]
metric_labels = ["Acuracia", "Revocacao", "Precisao", "F1-score"]
colors = ["#1f77b4", "#ff7f0e", "#2ca02c", "#d62728", "#9467bd", "#8c564b"]

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, mk, ml in zip(axes, metric_keys, metric_labels):
    means = [np.mean(results[n][0][mk]) for n in model_names]
    stds  = [np.std(results[n][0][mk])  for n in model_names]
    bars = ax.bar(short_names, means, yerr=stds, capsize=6,
                  color=colors, alpha=0.85, edgecolor="black")
    ax.set_title(ml, fontsize=12, fontweight="bold")
    ax.set_ylabel(ml)
    ax.set_ylim([0, 1.15])
    ax.grid(True, alpha=0.3, axis="y")
    for bar, m in zip(bars, means):
        ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.02,
                f"{m:.3f}", ha="center", fontsize=8)

plt.suptitle("Comparacao dos Modelos — Validacao Cruzada 10 Folds", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Distribuição das Métricas nos 10 Folds (Boxplots)

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()

for ax, mk, ml in zip(axes, metric_keys, metric_labels):
    data_box = [results[n][0][mk] for n in model_names]
    bp = ax.boxplot(data_box, labels=short_names, patch_artist=True)
    for patch, color in zip(bp["boxes"], colors):
        patch.set_facecolor(color)
        patch.set_alpha(0.7)
    ax.set_title(ml, fontsize=12, fontweight="bold")
    ax.set_ylabel(ml)
    ax.set_ylim([0, 1.1])
    ax.grid(True, alpha=0.3)

plt.suptitle("Distribuicao das Metricas por Fold (10-Fold CV)", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## Matrizes de Confusão

In [ ]:
classes = np.unique(y)
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
axes = axes.flatten()

for ax, name in zip(axes, model_names):
    _, yt, yp = results[name]
    cm = confusion_matrix(yt, yp, classes)
    plot_cm(cm, classes, name, ax)

plt.suptitle("Matrizes de Confusao (acumulado 10 folds)", fontsize=13, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()